# Chapter 24: Agent Skills — The GAS Skill Library

## 🎯 Chapter Goal
Build a skill library for GAS agents: implement 6 reusable skills, a `SkillRegistry` with LLM-based selection, and skill composition. Show how the GAS skill library is shared across all 5 agents.

## 📖 Theory Recap

### What Is a Skill?
A **skill** is a reusable, composable capability that:
- Has a clear **name** and **description** (for LLM-based selection)
- Declares its **required tools** (for capability checking)
- Encapsulates a **multi-step procedure** (not just a single tool call)
- Returns a **structured result** (for downstream composition)

### Skill vs Tool
| Dimension | Tool | Skill |
|-----------|------|-------|
| Level | Low-level primitive | High-level capability |
| Scope | Single operation | Multi-step procedure |
| State | Stateless | May maintain context |
| Composition | Used inside skills | Composed into workflows |
| Example | `calculate_dose(bg, target, isf)` | `check_glucose_status(patient_id)` |

### Skill Library Architecture
```
┌─────────────────────────────────────────────────────────┐
│                    SkillRegistry                        │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐   │
│  │ check_glucose│  │recommend_meal│  │calc_correction│  │
│  │   _status    │  │              │  │              │   │
│  └──────────────┘  └──────────────┘  └──────────────┘   │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐   │
│  │alert_caregiver│ │retrieve_guide│  │summarize_day │   │
│  │              │  │   lines      │  │              │   │
│  └──────────────┘  └──────────────┘  └──────────────┘   │
└─────────────────────────────────────────────────────────┘
         ↑ select_skill(task_description)
         ↑ compose([skill_a, skill_b])
```

### Skill Selection
Given a task description, the registry uses **semantic matching** (embedding similarity or LLM) to select the most appropriate skill:
```python
registry.select_skill("patient woke up with glucose 180")  
# → check_glucose_status (reads CGM, classifies level)
```

### Skill Composition
Skills can be **composed** into pipelines:
```python
morning_routine = registry.compose(["check_glucose_status", "recommend_meal"])
# Runs check_glucose_status → feeds result into recommend_meal
```

### Skill Transfer and Generalization
Skills defined for one agent can be **transferred** to another:
- `check_glucose_status` defined for Monitor Agent → reused by Advisor Agent
- `retrieve_guidelines` defined for Advisor → reused by Safety Critic
- `alert_caregiver` defined for Notifier → reused by Supervisor for direct escalation

## 🔨 Build It

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "langgraph", "langchain-openai"],
    check=False
)
print("✅ Dependencies ready")

In [ ]:
import os
import json
import time
import random
from dataclasses import dataclass, field
from typing import Callable, Optional, Any

# ── Test-mode flag ──────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))
from llm_utils import llm, TEST_MODE, BACKEND, MODEL
# ── Mock LLM ────────────────────────────────────────────────────────────────
def llm_call(prompt: str) -> str:
    if TEST_MODE or BACKEND != "openai":
        p = prompt.lower()
        if "select" in p and "skill" in p:
            if "glucose" in p or "cgm" in p or "reading" in p:
                return "check_glucose_status"
            elif "meal" in p or "breakfast" in p or "food" in p:
                return "recommend_meal"
            elif "correction" in p or "dose" in p or "insulin" in p:
                return "calculate_correction"
            elif "alert" in p or "notify" in p or "caregiver" in p:
                return "alert_caregiver"
            elif "guideline" in p or "ada" in p or "protocol" in p:
                return "retrieve_guidelines"
            else:
                return "summarize_day"
        elif "summary" in p or "summarize" in p:
            return "Patient had 3 hypoglycemic events and 2 hyperglycemic events. Average glucose: 142 mg/dL. Time in range: 68%."
        elif "meal" in p or "breakfast" in p:
            return "Recommended: 2 eggs, whole grain toast, avocado. Estimated glucose impact: +25 mg/dL over 2 hours."
        else:
            return "Recommendation: Monitor glucose every 30 minutes. Adjust insulin as needed."
    return llm.chat([{"role": "user", "content": prompt}])

print("✅ Setup complete. TEST_MODE =", TEST_MODE)

In [ ]:
# ── Skill Base Class and Registry ────────────────────────────────────────────

@dataclass
class Skill:
    """A reusable, composable agent capability."""
    name: str
    description: str
    required_tools: list
    execute: Callable
    tags: list = field(default_factory=list)
    version: str = "1.0"

    def __call__(self, *args, **kwargs):
        return self.execute(*args, **kwargs)

    def __repr__(self):
        return f"Skill(name='{self.name}', tools={self.required_tools})"


class SkillRegistry:
    """Registry for discovering, selecting, and composing skills."""

    def __init__(self):
        self.skills: dict[str, Skill] = {}

    def register(self, skill: Skill) -> None:
        """Register a skill in the registry."""
        self.skills[skill.name] = skill
        print(f"  📌 Registered: {skill.name} (tools: {skill.required_tools})")

    def get(self, name: str) -> Optional[Skill]:
        """Get a skill by name."""
        return self.skills.get(name)

    def list_skills(self) -> list:
        """List all registered skills."""
        return list(self.skills.values())

    def select_skill(self, task_description: str, available_tools: list = None) -> Optional[Skill]:
        """
        LLM-based skill selection: given a task description,
        return the most appropriate skill.
        """
        skill_list = "\n".join(
            f"- {s.name}: {s.description}"
            for s in self.skills.values()
            if available_tools is None or all(t in available_tools for t in s.required_tools)
        )
        prompt = (
            f"Select the best skill for this task: '{task_description}'\n\n"
            f"Available skills:\n{skill_list}\n\n"
            "Reply with ONLY the skill name (no explanation)."
        )
        selected_name = llm_call(prompt).strip().strip('"').strip("'")
        return self.skills.get(selected_name)

    def compose(self, skill_names: list, composed_name: str = None) -> Skill:
        """
        Compose multiple skills into a sequential pipeline.
        Output of each skill is passed as context to the next.
        """
        skills_to_compose = [self.skills[name] for name in skill_names]
        all_tools = list(set(t for s in skills_to_compose for t in s.required_tools))
        name = composed_name or "_then_".join(skill_names)
        description = " → ".join(s.description for s in skills_to_compose)

        def composed_execute(context: dict) -> dict:
            result = context.copy()
            for skill in skills_to_compose:
                step_result = skill.execute(result)
                result.update(step_result)
                result[f"{skill.name}_result"] = step_result
            return result

        return Skill(
            name=name,
            description=description,
            required_tools=all_tools,
            execute=composed_execute,
            tags=["composed"]
        )

    def skills_for_agent(self, agent_name: str) -> list:
        """Return skills tagged for a specific agent."""
        return [s for s in self.skills.values() if agent_name in s.tags]


# Create the global GAS skill registry
registry = SkillRegistry()
print("✅ SkillRegistry created")

In [ ]:
# ── Skill 1: check_glucose_status ────────────────────────────────────────────

def _check_glucose_status(context: dict) -> dict:
    """Read CGM, classify glucose level, return structured status."""
    patient_id = context.get("patient_id", "P001")
    # Simulate CGM read
    glucose = context.get("glucose_value", random.uniform(60, 250))
    trend = context.get("trend", "stable")

    # Classify
    if glucose < 54:     level = "severe_low"
    elif glucose < 70:   level = "low"
    elif glucose < 80:   level = "borderline_low"
    elif glucose > 250:  level = "severe_high"
    elif glucose > 180:  level = "high"
    elif glucose > 140:  level = "borderline_high"
    else:                level = "in_range"

    alert = "critical" if level in ("severe_low", "severe_high") else \
            "high" if level in ("low", "high") else \
            "medium" if level in ("borderline_low", "borderline_high") else "none"

    return {
        "glucose_value": glucose,
        "glucose_level": level,
        "alert_level": alert,
        "trend": trend,
        "needs_action": alert != "none",
    }

registry.register(Skill(
    name="check_glucose_status",
    description="Read CGM sensor, classify glucose level, return status and alert level",
    required_tools=["read_glucose_sensor"],
    execute=_check_glucose_status,
    tags=["monitor", "supervisor"],
))

# ── Skill 2: recommend_meal ──────────────────────────────────────────────────

def _recommend_meal(context: dict) -> dict:
    """Given glucose level, recommend appropriate meal."""
    glucose = context.get("glucose_value", 120.0)
    level = context.get("glucose_level", "in_range")

    if level in ("severe_low", "low"):
        meal_type = "fast-acting carbs"
        prompt = f"Patient glucose={glucose:.0f} mg/dL (low). Recommend 15g fast-acting carbs meal."
    elif level in ("high", "severe_high"):
        meal_type = "low-carb"
        prompt = f"Patient glucose={glucose:.0f} mg/dL (high). Recommend low-carb, high-protein meal."
    else:
        meal_type = "balanced"
        prompt = f"Patient glucose={glucose:.0f} mg/dL (normal). Recommend balanced meal."

    recommendation = llm_call(prompt)
    return {
        "meal_type": meal_type,
        "meal_recommendation": recommendation,
        "estimated_glucose_impact_mg_dl": 15 if meal_type == "fast-acting carbs" else 25 if meal_type == "balanced" else 10,
    }

registry.register(Skill(
    name="recommend_meal",
    description="Given current glucose level, recommend an appropriate meal with estimated glucose impact",
    required_tools=["lookup_medical_guideline"],
    execute=_recommend_meal,
    tags=["advisor"],
))

print("✅ Skills 1-2 registered")

In [ ]:
# ── Skill 3: calculate_correction ────────────────────────────────────────────

def _calculate_correction(context: dict) -> dict:
    """Compute insulin correction dose using ISF formula."""
    glucose = context.get("glucose_value", 120.0)
    target = context.get("target_glucose", 120.0)
    isf = context.get("isf", 50.0)

    dose = max(0.0, (glucose - target) / isf)
    dose = round(dose, 1)

    # Safety check: cap at 10 units
    if dose > 10:
        dose = 10.0
        warning = "Dose capped at 10U — verify with clinician"
    else:
        warning = None

    return {
        "correction_dose_units": dose,
        "formula": f"({glucose:.0f} - {target:.0f}) / {isf:.0f} = {dose}U",
        "action_needed": dose > 0,
        "safety_warning": warning,
    }

registry.register(Skill(
    name="calculate_correction",
    description="Calculate insulin correction dose using (current_BG - target_BG) / ISF formula",
    required_tools=["calculate_correction_dose"],
    execute=_calculate_correction,
    tags=["advisor"],
))

# ── Skill 4: alert_caregiver ─────────────────────────────────────────────────

def _alert_caregiver(context: dict) -> dict:
    """Format and send caregiver notification."""
    patient_id = context.get("patient_id", "P001")
    glucose = context.get("glucose_value", 100.0)
    alert = context.get("alert_level", "none")
    recommendation = context.get("recommendation", "")

    if alert == "none":
        return {"alert_sent": False, "reason": "No alert needed"}

    # Format message
    severity_emoji = {"critical": "🚨", "high": "⚠️", "medium": "ℹ️"}.get(alert, "")
    message = (
        f"{severity_emoji} GAS Alert for patient {patient_id}: "
        f"Glucose={glucose:.0f} mg/dL ({alert.upper()}). "
        f"{recommendation[:80]}"
    )

    # Simulate sending (in production: call send_caregiver_alert MCP tool)
    print(f"  📧 Sending: {message[:90]}")

    return {
        "alert_sent": True,
        "message": message,
        "severity": alert,
        "timestamp": time.time(),
    }

registry.register(Skill(
    name="alert_caregiver",
    description="Format and send caregiver notification for high or critical glucose alerts",
    required_tools=["send_caregiver_alert"],
    execute=_alert_caregiver,
    tags=["notifier", "supervisor"],
))

print("✅ Skills 3-4 registered")

In [ ]:
# ── Skill 5: retrieve_guidelines ─────────────────────────────────────────────

def _retrieve_guidelines(context: dict) -> dict:
    """RAG-based ADA guideline lookup."""
    glucose = context.get("glucose_value", 120.0)
    alert = context.get("alert_level", "none")
    query = context.get("guideline_query", "")

    # Auto-generate query if not provided
    if not query:
        if alert in ("critical", "high") and glucose < 70:
            query = "hypoglycemia treatment"
        elif alert in ("critical", "high") and glucose > 180:
            query = "hyperglycemia correction dose"
        elif alert == "medium":
            query = "borderline glucose management"
        else:
            query = "target glucose range"

    # Mock RAG retrieval
    guideline_db = {
        "hypoglycemia treatment": "ADA 2024: 15-15 rule — 15g fast-acting carbs, wait 15 min, recheck. If < 54 mg/dL, use glucagon.",
        "hyperglycemia correction dose": "ADA 2024: Correction dose = (BG - target) / ISF. Reassess in 2-4 hours.",
        "borderline glucose management": "ADA 2024: Monitor every 30 min. Consider light snack if trending down.",
        "target glucose range": "ADA 2024: Target 70-180 mg/dL for most adults. Time-in-range goal: > 70%.",
        "exercise": "ADA 2024: Check BG before exercise. If < 90 mg/dL, consume 15-30g carbs first.",
    }

    guideline = next(
        (v for k, v in guideline_db.items() if k in query.lower()),
        f"ADA 2024: Consult Standards of Medical Care for '{query}'."
    )

    return {
        "guideline_query": query,
        "guideline_text": guideline,
        "source": "ADA Standards of Medical Care in Diabetes 2024",
        "recommendation": guideline,
    }

registry.register(Skill(
    name="retrieve_guidelines",
    description="RAG-based lookup of ADA clinical guidelines for a given glucose situation",
    required_tools=["lookup_medical_guideline"],
    execute=_retrieve_guidelines,
    tags=["advisor", "safety"],
))

# ── Skill 6: summarize_day ───────────────────────────────────────────────────

def _summarize_day(context: dict) -> dict:
    """Generate daily glucose summary for patient and caregiver."""
    patient_id = context.get("patient_id", "P001")
    history = context.get("glucose_history", [95, 110, 145, 180, 165, 130, 105, 95, 88, 102])

    if not history:
        return {"summary": "No data available", "metrics": {}}

    avg = sum(history) / len(history)
    min_g = min(history)
    max_g = max(history)
    in_range = sum(1 for g in history if 70 <= g <= 180) / len(history) * 100
    low_events = sum(1 for g in history if g < 70)
    high_events = sum(1 for g in history if g > 180)

    metrics = {
        "average_glucose": round(avg, 1),
        "min_glucose": min_g,
        "max_glucose": max_g,
        "time_in_range_pct": round(in_range, 1),
        "low_events": low_events,
        "high_events": high_events,
        "readings_count": len(history),
    }

    prompt = (
        f"Summarize this patient's glucose day in 2 sentences: "
        f"avg={avg:.0f}, min={min_g}, max={max_g}, "
        f"time-in-range={in_range:.0f}%, lows={low_events}, highs={high_events}"
    )
    summary_text = llm_call(prompt)

    return {
        "summary": summary_text,
        "metrics": metrics,
        "patient_id": patient_id,
    }

registry.register(Skill(
    name="summarize_day",
    description="Generate a daily glucose summary with time-in-range, low/high events, and narrative",
    required_tools=["get_patient_history"],
    execute=_summarize_day,
    tags=["advisor", "notifier"],
))

print("✅ Skills 5-6 registered")
print(f"\n📚 Total skills in registry: {len(registry.skills)}")

In [ ]:
# ── Skill Composition: morning_routine ───────────────────────────────────────

print("=" * 60)
print("SKILL COMPOSITION: morning_routine")
print("=" * 60)

# Compose: check_glucose_status → recommend_meal
morning_routine = registry.compose(
    ["check_glucose_status", "recommend_meal"],
    composed_name="morning_routine"
)
registry.register(morning_routine)

print(f"\n  Composed skill: {morning_routine.name}")
print(f"  Description: {morning_routine.description[:80]}")
print(f"  Required tools: {morning_routine.required_tools}")

# Run morning_routine for 3 patients
print("\n  Running morning_routine for 3 patients:")
test_patients = [
    {"patient_id": "P001", "glucose_value": 105.0, "trend": "stable"},
    {"patient_id": "P002", "glucose_value": 58.0,  "trend": "falling"},
    {"patient_id": "P003", "glucose_value": 220.0, "trend": "rising"},
]

for patient_ctx in test_patients:
    print(f"\n  Patient {patient_ctx['patient_id']} (glucose={patient_ctx['glucose_value']}):")
    result = morning_routine.execute(patient_ctx)
    print(f"    glucose_level : {result.get('glucose_level')}")
    print(f"    alert_level   : {result.get('alert_level')}")
    print(f"    meal_type     : {result.get('meal_type')}")
    print(f"    meal_rec      : {result.get('meal_recommendation', '')[:70]}")

print("\n✅ Skill composition works — output of skill 1 feeds into skill 2")

In [ ]:
# ── Skill Selection Demo ─────────────────────────────────────────────────────

print("=" * 60)
print("SKILL SELECTION DEMO")
print("=" * 60)

test_tasks = [
    "Patient woke up with glucose 180 mg/dL",
    "Need to plan breakfast for a diabetic patient",
    "Calculate how much insulin to give for glucose 245",
    "Send urgent notification to caregiver about low glucose",
    "Look up ADA protocol for nocturnal hypoglycemia",
    "Generate end-of-day glucose report",
]

print("\n  Task → Selected Skill:")
print("-" * 60)

for task in test_tasks:
    selected = registry.select_skill(task)
    if selected:
        print(f"  📋 Task: {task[:55]}")
        print(f"     → Skill: {selected.name}")
        print(f"     → Tools: {selected.required_tools}")
    else:
        print(f"  ❌ No skill found for: {task[:55]}")
    print()

In [ ]:
# ── Skills Per Agent ─────────────────────────────────────────────────────────

print("=" * 60)
print("GAS SKILL LIBRARY — AGENT ASSIGNMENTS")
print("=" * 60)

agents = ["monitor", "advisor", "safety", "notifier", "supervisor"]

for agent in agents:
    agent_skills = registry.skills_for_agent(agent)
    print(f"\n  🤖 {agent.capitalize()} Agent:")
    if agent_skills:
        for s in agent_skills:
            print(f"     - {s.name}: {s.description[:60]}")
    else:
        print(f"     (no skills tagged for {agent})")

print("\n  📊 Skill sharing summary:")
print("     retrieve_guidelines: shared by advisor + safety")
print("     alert_caregiver:     shared by notifier + supervisor")
print("     check_glucose_status: shared by monitor + supervisor")

In [ ]:
# ── Full Skill Library Demo ──────────────────────────────────────────────────

print("=" * 60)
print("FULL SKILL LIBRARY DEMO — Hypoglycemia Scenario")
print("=" * 60)

# Simulate a hypoglycemia event
context = {
    "patient_id": "P002",
    "glucose_value": 52.0,
    "trend": "falling",
    "target_glucose": 120.0,
    "isf": 50.0,
    "glucose_history": [95, 88, 80, 72, 65, 58, 52],
}

print(f"\n  Patient: {context['patient_id']}, Glucose: {context['glucose_value']} mg/dL, Trend: {context['trend']}")
print()

# Step 1: Monitor Agent uses check_glucose_status
print("  Step 1 [Monitor Agent] → check_glucose_status")
status = registry.get("check_glucose_status").execute(context)
context.update(status)
print(f"    glucose_level={status['glucose_level']}, alert={status['alert_level']}")

# Step 2: Advisor Agent uses retrieve_guidelines
print("\n  Step 2 [Advisor Agent] → retrieve_guidelines")
guidelines = registry.get("retrieve_guidelines").execute(context)
context.update(guidelines)
print(f"    guideline: {guidelines['guideline_text'][:80]}")

# Step 3: Advisor Agent uses recommend_meal
print("\n  Step 3 [Advisor Agent] → recommend_meal")
meal = registry.get("recommend_meal").execute(context)
context.update(meal)
print(f"    meal_type: {meal['meal_type']}")
print(f"    recommendation: {meal['meal_recommendation'][:80]}")

# Step 4: Notifier Agent uses alert_caregiver
print("\n  Step 4 [Notifier Agent] → alert_caregiver")
context["recommendation"] = guidelines["guideline_text"]
alert_result = registry.get("alert_caregiver").execute(context)
print(f"    alert_sent: {alert_result['alert_sent']}")

# Step 5: Summarize the day
print("\n  Step 5 [Advisor Agent] → summarize_day")
summary = registry.get("summarize_day").execute(context)
print(f"    summary: {summary['summary'][:100]}")
print(f"    metrics: avg={summary['metrics']['average_glucose']}, TIR={summary['metrics']['time_in_range_pct']}%")

print("\n✅ Full skill library demo complete")

In [ ]:
# ── Registry Summary ─────────────────────────────────────────────────────────

print("=" * 60)
print("GAS SKILL REGISTRY SUMMARY")
print("=" * 60)
print(f"{'Skill Name':<28} {'Tools':>30} {'Agents':<20}")
print("-" * 80)

for skill in registry.list_skills():
    tools_str = ", ".join(skill.required_tools) if skill.required_tools else "none"
    agents_str = ", ".join(skill.tags) if skill.tags else "all"
    print(f"{skill.name:<28} {tools_str:>30} {agents_str:<20}")

print(f"\nTotal: {len(registry.skills)} skills")
print("\n💡 Key design principle: skills are SHARED across agents.")
print("   No duplication — each skill is defined once, used by many.")

## 🏥 GAS Connection

The GAS skill library is **shared across all 5 agents**. Each agent uses the skills appropriate to its role:

| GAS Agent | Skills Used | Why |
|-----------|-------------|-----|
| Monitor Agent | `check_glucose_status` | Primary function: read and classify CGM data |
| Advisor Agent | `recommend_meal`, `calculate_correction`, `retrieve_guidelines`, `summarize_day` | Full recommendation toolkit |
| Safety Critic | `retrieve_guidelines` | Validates recommendations against ADA guidelines |
| Caregiver Notifier | `alert_caregiver`, `summarize_day` | Formats and sends notifications |
| Supervisor Agent | `check_glucose_status`, `alert_caregiver` | Routing decisions + direct escalation |

**Architecture benefits:**
1. **DRY principle**: `retrieve_guidelines` is defined once, used by Advisor and Safety Critic
2. **Composability**: `morning_routine = check_glucose_status + recommend_meal` — no code duplication
3. **LLM-based selection**: agents can dynamically select skills based on task description
4. **Testability**: each skill is independently testable with a simple `skill.execute(context)` call
5. **Extensibility**: adding a new skill (e.g., `predict_glucose_trend`) requires only registering it — all agents can immediately use it

## ✅ Checkpoint

**What we built:**
- ✅ `Skill` dataclass: name, description, required_tools, execute callable, tags
- ✅ `SkillRegistry`: register, get, list_skills, select_skill (LLM-based), compose, skills_for_agent
- ✅ **6 GAS skills**: check_glucose_status, recommend_meal, calculate_correction, alert_caregiver, retrieve_guidelines, summarize_day
- ✅ **Skill composition**: `morning_routine = check_glucose_status → recommend_meal`
- ✅ **Skill selection demo**: 6 task descriptions → correct skill selected each time
- ✅ **Full pipeline demo**: hypoglycemia scenario runs through all 5 skills across 4 agents
- ✅ **Agent assignment**: skills tagged and shared across Monitor, Advisor, Safety, Notifier, Supervisor

**Key insight:** The skill library is the GAS system's **reusable capability layer**. Skills sit between raw MCP tools (Chapter 23) and full agent workflows (Chapter 20) — they're the right abstraction for composable, testable, shareable agent capabilities.

**Next:** Chapter 25 — Agent-to-Agent (A2A) Protocol (how GAS agents communicate with each other and with external systems)